# M2-B1 — Audit de qualité et effet du nettoyage

Les **outils** sont fournis (rapport de qualité, comparaison des modèles). Les **constats**, les **décisions**
et leur **interprétation** sont à toi : réponds dans les cellules ✏️.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date, r.reservation_id
"""), engine, parse_dates=["arrival_date", "reservation_status_date"])
df.shape

## Tâche 2 — L'audit

In [ ]:
# Bornes « raisonnables » par colonne numérique : hypothèses de départ, à discuter
BORNES = {
    "lead_time": (0, 730),
    "adults": (1, 10),
    "children": (0, 10),
    "babies": (0, 5),
    "stays_in_weekend_nights": (0, 20),
    "stays_in_week_nights": (0, 50),
    "adr": (0, 1000),
}


def rapport_qualite(d: pd.DataFrame) -> pd.DataFrame:
    """Manquants et valeurs hors bornes, colonne par colonne."""
    lignes = []
    for col in d.columns:
        info = {"colonne": col, "type": str(d[col].dtype),
                "manquants": int(d[col].isna().sum()), "valeurs_distinctes": d[col].nunique()}
        if col in BORNES:
            bas, haut = BORNES[col]
            info["hors_bornes"] = int(((d[col] < bas) | (d[col] > haut)).sum())
        lignes.append(info)
    return pd.DataFrame(lignes).set_index("colonne")


rapport = rapport_qualite(df)
rapport[(rapport["manquants"] > 0) | (rapport.get("hors_bornes", 0) > 0)]

In [ ]:
# Doublons : réservations identiques sur toutes les colonnes, sauf l'identifiant
doublons = df.drop(columns="reservation_id").duplicated(keep="first")
print(doublons.sum(), "lignes en double")

### ✏️ Ce que le rapport montre

Pour chaque problème : combien de lignes, et à quoi elles ressemblent (affiche quelques exemples).
Les doublons sont-ils forcément des erreurs ? 

Sur les **78 703 réservations**, le rapport relève :

- **`lead_time` : 1 hors borne** (la borne haute choisie est 730 jours). La réservation `81` a un délai de **737 jours** ; c'est un cas extrême à vérifier.
- **`adults` : 285 hors bornes** : **273** réservations ont 0 adulte et **12** en ont plus de 10. Par exemple, les réservations `1960` et `2252` indiquent 0 adulte.
- **`children` : 4 valeurs manquantes**. Par exemple, la réservation `2941` indique 2 adultes, mais le nombre d'enfants est inconnu.
- **`babies` : 2 hors bornes** : les réservations `13760` et `22710` indiquent respectivement 9 et 10 bébés, au-dessus de la borne choisie de 5.
- **`country` : 405 valeurs manquantes**. Par exemple, les réservations `110` et `113` ont des voyageurs mais aucun pays renseigné.
- **`agent` : 10 676 valeurs manquantes**. Les réservations `80` et `82` sont des réservations directes, ce qui peut expliquer l'absence d'agent.
- **`company` : 74 216 valeurs manquantes**. Par exemple, les réservations `1` et `2` n'ont pas de société indiquée. Cela peut être normal si la réservation n'est pas liée à une entreprise.
- **`adr` : 1 hors borne** : la réservation `31829` a un ADR de **5 400** pour 2 adultes et une nuit ; elle est annulée. C'est un outlier manifeste à vérifier.

Les champs `agent` et `company` manquants ne sont donc pas nécessairement des erreurs : ils peuvent être sans objet selon le canal ou le type de réservation. Les bornes sont aussi des hypothèses d'audit, pas des règles métier confirmées.

Le contrôle des doublons compare toutes les colonnes sauf `reservation_id` et trouve **22 999 lignes répétées** (les répétitions au-delà de la première). Par exemple, les réservations `6` à `10` ont les mêmes autres valeurs. Ce ne sont pas forcément des erreurs : plusieurs réservations distinctes peuvent avoir exactement les mêmes caractéristiques. Il faut un identifiant source ou une règle métier pour conclure qu'il s'agit de copies.

### ✏️ Ce que le rapport ne voit pas

Le rapport ne connaît que des bornes colonne par colonne. Cherche les **incohérences métier** : des combinaisons
de valeurs impossibles ou suspectes pour un hôtel. Au moins deux, chiffrées. Écris le code dans les cellules
ci-dessous.



In [ ]:
nuits = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
enfants = df["children"].fillna(0)
occupants = df["adults"].fillna(0) + enfants + df["babies"].fillna(0)

anomalies = {
    "Aucun occupant et aucune nuit": df["adults"].eq(0) & occupants.eq(0) & nuits.eq(0),
    "Aucun occupant malgré une ou plusieurs nuits": occupants.eq(0) & nuits.gt(0),
    "Enfants/bébés sans adulte": df["adults"].eq(0) & (enfants + df["babies"].fillna(0)).gt(0),
    "Zéro nuit avec statut Check-Out": nuits.eq(0) & df["reservation_status"].eq("Check-Out"),
    "ADR nul avec nuitée et statut Check-Out": df["adr"].eq(0) & nuits.gt(0) & df["reservation_status"].eq("Check-Out"),
}

colonnes_exemple = [
    "reservation_id", "hotel", "reservation_status", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies", "adr",
]
for nom, masque in anomalies.items():
    print(f"{nom} : {int(masque.sum())} lignes")
    print("Statuts :", df.loc[masque, "reservation_status"].value_counts().to_dict())
    display(df.loc[masque, colonnes_exemple].head(5))


## Tâche 3 — Les décisions

| Anomalie | Lignes | Décision (supprimer / corriger / imputer / garder et signaler) | Justification | Question au client ? |
|---|---|---|---|---|
| `children` manquant | 4 | Imputer à 0 (règle fournie) | Très peu de cas ; en l'absence d'indication, 0 enfant est l'hypothèse la plus prudente. Garder le compte des lignes imputées. | Confirmer que vide signifie bien « aucun enfant ». |
| `country` manquant | 405 | Garder et signaler ; utiliser `Unknown` seulement pour le modèle | Le pays est inconnu, pas nécessairement erroné ; supprimer ces lignes ferait perdre des réservations. | Le pays est-il facultatif ou doit-il être récupéré à la source ? |
| `agent` manquant | 10 676 | Garder | Des réservations directes peuvent ne pas avoir d'agent ; ne pas inventer de code agent. | Une valeur vide signifie-t-elle « réservation directe » ? |
| `company` manquant | 74 216 | Garder et signaler | Une société peut être sans objet pour une réservation individuelle ; ne pas imputer un identifiant arbitraire. | Le champ est-il vide pour les clients non affiliés à une entreprise ? |
| `lead_time` hors borne (737 jours) | 1 | Remplacer par manquant ; la médiane du pipeline l'impute | Évite d'utiliser une valeur au-delà de la borne d'audit sans inventer un délai précis. | Confirmer la fenêtre maximale de réservation autorisée. |
| `adults` hors [1, 10] | 285 (273 à 0, 12 au-dessus de 10) | Supprimer | Le modèle n'est pas entraîné sur des réservations dont le nombre d'adultes est hors de la plage retenue. | Vérifier si la limite `[1, 10]` doit aussi s'appliquer aux réservations de groupe. |
| `babies` hors [0, 5] (valeurs 9 et 10) | 2 | Remplacer par manquant | Évite de choisir arbitrairement le nombre de bébés ; cette colonne n'est pas utilisée par le modèle. | Vérifier si ces valeurs sont des erreurs de saisie ou de catégorie. |
| Aucun occupant et aucune nuit | 36 (31 `Check-Out`, 5 `Canceled`) | Supprimer | Aucune personne ni nuitée n'est enregistrée ; on ne garde pas ces lignes dans l'entraînement. | À confirmer si ces réservations correspondent à des enregistrements techniques. |
| Zéro nuit avec statut `Check-Out` | 547 (dont 31 déjà comptées ci-dessus) | Supprimer | Le statut indique un séjour terminé sans aucune nuitée. | Confirmer qu'un séjour à la journée ne peut pas avoir le statut `Check-Out`. |
| ADR nul avec nuitée et statut `Check-Out` | 796 | Garder et signaler | Une gratuité ou un tarif inclus est possible ; un prix nul n'est pas forcément une anomalie. | Ces séjours sont-ils gratuits, offerts ou facturés dans un autre champ ? |
| ADR supérieur à 1 000 (valeur 5 400) | 1 | Remplacer par manquant ; la médiane du pipeline l'impute | Une valeur extrême risquerait de fausser l'échelle du prix ; aucun tarif de remplacement n'est inventé. | Vérifier si 5 400 est réel ou une erreur de saisie/unité. |
| Lignes identiques hors `reservation_id` | 22 999 répétitions dans le brut | Garder et signaler | Des réservations distinctes peuvent partager les mêmes valeurs hors identifiant ; les supprimer pourrait éliminer de vrais séjours. | Ces lignes correspondent-elles à des réservations légitimes différentes ou à des doublons d'import ? Quelle règle fiable permet de les distinguer ? |

Les critères se chevauchent : les 36 réservations sans occupant ni nuit comprennent 31 des 547 `Check-Out` sans nuit et sont aussi incluses parmi les adultes hors bornes. Ne pas additionner directement les effectifs bruts, car certaines lignes relèvent de plusieurs anomalies.

## Tâche 6 — Comparer v2.0.0 et v2.1.0

À faire **après** avoir branché le nettoyage et produit la v2.1.0 avec `python train.py`.
La fonction de comparaison est fournie : c'est le **choix du jeu de test** qui t'appartient.

In [ ]:
import joblib
from sklearn.metrics import f1_score, precision_score, recall_score

import train as T
from features import INPUT_FEATURES


def comparer(modeles: dict, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    lignes = []
    for nom, modele in modeles.items():
        pred = modele.predict(X)
        lignes.append({"modèle": nom, "f1": f1_score(y, pred), "rappel": recall_score(y, pred),
                       "précision": precision_score(y, pred), "n": len(y)})
    return pd.DataFrame(lignes).set_index("modèle").round(3)


modeles = {
    "v2.0.0 (données brutes)": joblib.load(ROOT / "model" / "cancellation_model_v2.0.0.joblib"),
    "v2.1.0 (données nettoyées)": joblib.load(ROOT / "model" / "cancellation_model_v2.1.0.joblib"),
}

brut = T.load_data(engine)
_, test_brut = T.split_train_test(brut)
comparer(modeles, test_brut[INPUT_FEATURES], test_brut[T.TARGET])

In [ ]:
from nettoyage import nettoyer

donnees_nettoyees, compte_rendu_nettoyage = nettoyer(brut)
_, test_nettoye = T.split_train_test(donnees_nettoyees)
print(f"Test brut : {len(test_brut)} lignes | test nettoyé : {len(test_nettoye)} lignes")
print("Nettoyage :\n", compte_rendu_nettoyage.to_string(index=False))
comparer(modeles, test_nettoye[INPUT_FEATURES], test_nettoye[T.TARGET])

### ✏️ Interprétation

1. **Le score baisse légèrement pour v2.1 sur les deux populations.** Sur le test brut (19 911 lignes), le F1 passe de 0,734 à 0,725, le rappel de 0,754 à 0,741 et la précision de 0,715 à 0,710. Sur le test nettoyé (19 696 lignes), le F1 passe de 0,736 à 0,728, le rappel de 0,755 à 0,742 et la précision de 0,718 à 0,714. Cette différence vient du nouvel entraînement sur des données nettoyées et n'est pas, à elle seule, une preuve que le modèle est meilleur ou pire en production. Une baisse peut être acceptable si elle évite des données invalides, mais il faut comparer l'impact métier des faux positifs et faux négatifs ; ici, les métriques indiquent surtout qu'il faut poursuivre l'évaluation.

2. **801 lignes sont retirées de l'historique** : 36 sans occupant ni nuit, 516 `Check-Out` sans nuit après le premier filtre, puis 249 avec `adults` hors `[1, 10]` après les filtres précédents. Le jeu passe de 78 703 à 77 902 lignes ; dans le test temporel, 215 lignes sur 19 911 sont retirées. Les 22 999 lignes identiques hors `reservation_id` sont conservées. Le modèle ne sait pas refuser une demande : sans validation API, il renverra quand même une prédiction pour une réservation similaire, même si elle sort du domaine d'entraînement. L'API doit donc signaler ou rejeter les valeurs invalides plutôt que de présenter aveuglément le résultat comme fiable.

3. **Oui, mais seulement les règles disponibles au moment de la prédiction**, et exactement de la même façon qu'à l'entraînement : par exemple, imputer `children` manquant à 0 et valider `adults` dans `[1, 10]`. Les règles utilisant `reservation_status = Check-Out` reposent sur le résultat final du séjour et ne peuvent pas être appliquées à une nouvelle réservation. De même, ne pas dédupliquer sur les champs descriptifs : des réservations différentes peuvent être identiques hors `reservation_id`. L'API doit assurer l'unicité de l'identifiant et demander au client si plusieurs lignes identiques sont légitimes ou des doublons d'import.

4. **Le test brut ressemble le plus aux entrées réelles de l'API** : il conserve les réservations avant les règles historiques. Je retiens ce test pour la décision principale, car il évalue les deux modèles sur les mêmes 19 911 lignes ; je présente aussi le test nettoyé (19 696 lignes) en complément pour mesurer les résultats sur la population conservée après les contrôles. En production, la comparaison idéale utilisera un jeu brut récent et appliquera seulement les mêmes validations et transformations que l'API, sans filtres fondés sur le statut final.